In [15]:
import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import wandb

ENTITY = "ml-for-data-analytics-project"
PROJECT = "energy-forecasting"
PREDICTIONS_KEY = "predictions"

api = wandb.Api()
runs = api.runs(f"{ENTITY}/{PROJECT}")
print(f"Found {len(runs)} runs in {ENTITY}/{PROJECT}")

Found 20 runs in ml-for-data-analytics-project/energy-forecasting


In [16]:
runs[3].summary.get(PREDICTIONS_KEY).get("path")


'media/table/predictions_49_848054a8cc6d04d06c7a.table.json'

In [17]:
def _extract_table_path_from_summary(summary_value):
    if isinstance(summary_value, dict):
        path = summary_value.get("path")
        if isinstance(path, str) and path.endswith(".table.json"):
            return path
    return None


def _download_predictions_table(run, key=PREDICTIONS_KEY):
    candidate_paths = []
    summary_path = _extract_table_path_from_summary(run.summary.get(key))
    if summary_path:
        candidate_paths.append(summary_path)

    try:
        for f in run.files():
            name = getattr(f, "name", "")
            if name.endswith(".table.json") and key.lower() in name.lower():
                candidate_paths.append(name)
    except Exception:
        pass

    seen = set()
    for rel_path in candidate_paths:
        if rel_path in seen:
            continue
        seen.add(rel_path)

        try:
            downloaded = run.file(rel_path).download(
                root=tempfile.gettempdir(),
                replace=True,
            )
            with open(downloaded.name, "r", encoding="utf-8") as fp:
                payload = json.load(fp)

            if isinstance(payload, dict) and "columns" in payload and "data" in payload:
                df = pd.DataFrame(payload["data"], columns=payload["columns"])
                return df, rel_path
        except Exception:
            continue

    return None, None


all_predictions = []
run_level_stats = []
missing_predictions = []

for run in runs:
    df_pred, source_path = _download_predictions_table(run)
    if df_pred is None or df_pred.empty:
        missing_predictions.append(run.id)
        continue

    df_pred = df_pred.copy()
    df_pred["run_id"] = run.id
    df_pred["run_name"] = run.name
    df_pred["run_state"] = run.state
    df_pred["table_path"] = source_path

    all_predictions.append(df_pred)

    actual_col = "actual_kWh" if "actual_kWh" in df_pred.columns else None
    pred_col = "predicted_kWh" if "predicted_kWh" in df_pred.columns else None

    if actual_col and pred_col:
        y_true = pd.to_numeric(df_pred[actual_col], errors="coerce")
        y_pred = pd.to_numeric(df_pred[pred_col], errors="coerce")

        mae = float(np.mean(np.abs(y_true - y_pred)))
        rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
        non_zero = y_true != 0
        mape = (
            float(np.mean(np.abs((y_true[non_zero] - y_pred[non_zero]) / y_true[non_zero])) * 100)
            if bool(non_zero.any())
            else np.nan
        )
    else:
        mae = np.nan
        rmse = np.nan
        mape = np.nan

    run_level_stats.append(
        {
            "run_id": run.id,
            "run_name": run.name,
            "state": run.state,
            "created_at": run.created_at,
            "table_rows": int(len(df_pred)),
            "table_cols": int(df_pred.shape[1]),
            "table_path": source_path,
            "summary_val/mae": run.summary.get("val/mae"),
            "summary_val/rmse": run.summary.get("val/rmse"),
            "summary_val/mse": run.summary.get("val/mse"),
            "table_mae": mae,
            "table_rmse": rmse,
            "table_mape_pct": mape,
        }
    )

print(f"Runs with prediction tables: {len(run_level_stats)}")
print(f"Runs without prediction tables: {len(missing_predictions)}")

Runs with prediction tables: 17
Runs without prediction tables: 3


In [18]:
df_pred

,date,actual_kWh,predicted_kWh,error,run_id,run_name,run_state,table_path
0,2017-01-01T00:00:00,30.293125,33.644434,-3.351309,rg0maq7x,encdec_optuna_best_lr0.00071_dr0.1_hs128_bs7_e...,finished,media/table/predictions_90_d379f07a4752ad71758...
1,2017-01-02T00:00:00,25.402792,26.865369,-1.462577,rg0maq7x,encdec_optuna_best_lr0.00071_dr0.1_hs128_bs7_e...,finished,media/table/predictions_90_d379f07a4752ad71758...
2,2017-01-03T00:00:00,29.077583,26.641488,2.436095,rg0maq7x,encdec_optuna_best_lr0.00071_dr0.1_hs128_bs7_e...,finished,media/table/predictions_90_d379f07a4752ad71758...
3,2017-01-04T00:00:00,36.197583,31.109379,5.088204,rg0maq7x,encdec_optuna_best_lr0.00071_dr0.1_hs128_bs7_e...,finished,media/table/predictions_90_d379f07a4752ad71758...
4,2017-01-05T00:00:00,36.335583,31.762572,4.573011,rg0maq7x,encdec_optuna_best_lr0.00071_dr0.1_hs128_bs7_e...,finished,media/table/predictions_90_d379f07a4752ad71758...
...,...,...,...,...,...,...,...,...
60,2017-03-02T00:00:00,15.813333,20.635101,-4.821767,rg0maq7x,encdec_optuna_best_lr0.00071_dr0.1_hs128_bs7_e...,finished,media/table/predictions_90_d379f07a4752ad71758...
61,2017-03-03T00:00:00,13.449167,13.384912,0.064255,rg0maq7x,encdec_optuna_best_lr0.00071_dr0.1_hs128_bs7_e...,finished,media/table/predictions_90_d379f07a4752ad71758...
62,2017-03-04T00:00:00,11.324375,11.861757,-0.537382,rg0maq7x,encdec_optuna_best_lr0.00071_dr0.1_hs128_bs7_e...,finished,media/table/predictions_90_d379f07a4752ad71758...
63,2017-03-05T00:00:00,13.648833,13.636829,0.012005,rg0maq7x,encdec_optuna_best_lr0.00071_dr0.1_hs128_bs7_e...,finished,media/table/predictions_90_d379f07a4752ad71758...


In [19]:
combined_predictions_df = (
    pd.concat(all_predictions, ignore_index=True)
    if all_predictions
    else pd.DataFrame()
)

prediction_summary_df = pd.DataFrame(run_level_stats).sort_values(
    by="table_rmse",
    ascending=True,
    na_position="last",
).reset_index(drop=True)

combined_path = Path("data/processed/wandb_all_run_predictions.csv")
summary_path = Path("data/processed/wandb_prediction_summary_by_run.csv")
combined_path.parent.mkdir(parents=True, exist_ok=True)

if not combined_predictions_df.empty:
    combined_predictions_df.to_csv(combined_path, index=False)
prediction_summary_df.to_csv(summary_path, index=False)

print(f"Saved summary table to: {summary_path}")
if not combined_predictions_df.empty:
    print(f"Saved combined predictions table to: {combined_path}")
if missing_predictions:
    
    preview_missing = missing_predictions[:10]
    suffix = " ..." if len(missing_predictions) > 10 else ""
    print(f"Missing predictions table in runs: {preview_missing}{suffix}")

prediction_summary_df

Saved summary table to: data/processed/wandb_prediction_summary_by_run.csv
Saved combined predictions table to: data/processed/wandb_all_run_predictions.csv
Missing predictions table in runs: ['pni4b3j8', 'yynhasqk', 'm59aj3kq']


,run_id,run_name,state,created_at,table_rows,table_cols,table_path,summary_val/mae,summary_val/rmse,summary_val/mse,table_mae,table_rmse,table_mape_pct
0,vx6nj6rb,"LSTMModel0_lr0.001_seq[1, 2]",finished,2026-03-30T16:19:24Z,63,8,media/table/predictions_49_f034b1cbb669941840c...,0.041127,0.051586,0.002661,1.402818,1.759589,5.382815
1,4pv5mdmd,"LSTMModel0_lr0.001_seq[1, 2]_dropout",finished,2026-03-30T16:27:40Z,63,8,media/table/predictions_49_157afef3f2b36b452ea...,0.049643,0.062289,0.003880,1.693316,2.124662,6.292367
2,rg0maq7x,encdec_optuna_best_lr0.00071_dr0.1_hs128_bs7_e...,finished,2026-05-01T15:09:35Z,65,8,media/table/predictions_90_d379f07a4752ad71758...,0.102179,0.132278,0.017497,2.347897,3.097904,8.849394
3,csq3aeg9,encdec_optuna_best_lr0.0008500000000000001_dr0...,finished,2026-05-01T14:32:24Z,65,8,media/table/predictions_119_66bf8d81bdfcae35f4...,0.099032,0.131313,0.017243,2.519780,3.314889,9.135008
4,i6nfjq31,"DECOMPOSE_ARIMA_order(1, 1, 2)_p7",finished,2026-04-24T22:43:43Z,65,8,media/table/predictions_0_5cb9fff2dfd65cf08277...,2.727446,3.352765,11.241033,2.727446,3.352765,10.976243
5,znxzbbdy,"SARIMA_order(0, 1, 3)_sorder(1, 1, 1, 7)",finished,2026-04-24T22:44:47Z,65,8,media/table/predictions_0_5001ef567df28d5c6d0e...,2.728761,3.410608,11.632247,2.728761,3.410608,11.022812
6,fwg0uzd9,"SARIMAX_order(0, 1, 3)_sorder(0, 1, 1, 7)",finished,2026-04-24T22:03:15Z,65,8,media/table/predictions_0_8db6d0f64cfd5a839e11...,2.725984,3.420405,11.699173,2.725984,3.420405,11.115014
7,oowvfmc3,encdec_optuna_best_lr0.0005_dr0.1_hs100_bs8_ep140,finished,2026-05-01T14:11:44Z,65,8,media/table/predictions_113_cf907b0bb654cd8376...,0.052137,0.073880,0.005458,2.818975,3.656776,10.443073
8,vjh4lfko,"LSTMModel0_lr0.001_seq[1, 2]",finished,2026-04-29T20:44:27Z,65,8,media/table/predictions_49_50409123e2028d5b83c...,0.090661,0.112120,0.012571,3.092417,3.824362,11.921595
9,n3viomoq,LSTMAttentionModel_lr0.001,finished,2026-05-01T09:27:01Z,65,8,media/table/predictions_49_7ff2dc8a68e7ea1528c...,0.090309,0.114665,0.013148,3.080398,3.911165,11.987206


In [20]:
import os
from datetime import datetime, timezone

# Fallback so this cell can run independently after kernel restart.
if "prediction_summary_df" not in globals() or prediction_summary_df.empty:
    summary_path = Path("data/processed/wandb_prediction_summary_by_run.csv")
    prediction_summary_df = pd.read_csv(summary_path)

if "combined_predictions_df" not in globals() or combined_predictions_df.empty:
    combined_path = Path("data/processed/wandb_all_run_predictions.csv")
    if combined_path.exists():
        combined_predictions_df = pd.read_csv(combined_path)
    else:
        combined_predictions_df = pd.DataFrame()

active_run = getattr(wandb, "run", None)
if active_run is not None:
    try:
        wandb.finish(quiet=True)
    except Exception:
        pass

init_kwargs = {
    "entity": ENTITY,
    "project": PROJECT,
    "job_type": "analysis",
    "name": f"prediction-summary-upload-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}",
}

wandb_mode = (os.environ.get("WANDB_MODE") or "").strip().lower()
if wandb_mode in {"offline", "disabled"}:
    print("W&B is configured for offline mode in this environment.")
    init_kwargs["mode"] = "offline"

try:
    upload_run = wandb.init(**init_kwargs)
except (ConnectionResetError, OSError) as exc:
    print(f"W&B init failed ({exc}); retrying in offline mode.")
    try:
        wandb.finish(quiet=True)
    except Exception:
        pass
    init_kwargs["mode"] = "offline"
    upload_run = wandb.init(**init_kwargs)

upload_run.log({
    "prediction_summary_by_run": wandb.Table(dataframe=prediction_summary_df),
})

artifact = wandb.Artifact(
    name="prediction-summary-by-run",
    type="dataset",
    description="Summary and combined prediction tables aggregated from all project runs.",
)
artifact.add_file(str(summary_path))

if "combined_path" in globals() and Path(combined_path).exists():
    artifact.add_file(str(combined_path))

upload_run.log_artifact(artifact)
upload_run.finish()

upload_target = upload_run.url or getattr(upload_run, "dir", None) or "local W&B run"
print(f"Uploaded table to run: {upload_target}")
print("Logged table key: prediction_summary_by_run")
print("Logged artifact name: prediction-summary-by-run")

Uploaded table to run: https://wandb.ai/ml-for-data-analytics-project/energy-forecasting/runs/0xhu4nfe
Logged table key: prediction_summary_by_run
Logged artifact name: prediction-summary-by-run


In [21]:

df = pd.read_csv("/Users/tomasz/Downloads/wandb_export_2026-05-01T14_30_52.528+02_00.csv")
mae = float(np.mean(np.abs(df.error)))
mse = float(np.mean(df.error ** 2))
rmse = float(np.sqrt(mse))
print(f"SEAS ARIMA MAE: {mae:.4f}, MSE: {mse:.4f}, RMSE: {rmse:.4f}")

SEAS ARIMA MAE: 2.7274, MSE: 11.2410, RMSE: 3.3528


In [22]:
df = pd.read_csv("/Users/tomasz/Downloads/wandb_export_2026-05-01T16_23_22.630+02_00.csv")
mae = float(np.mean(np.abs(df.error)))
mse = float(np.mean(df.error ** 2))
rmse = float(np.sqrt(mse))
print(f"Current MAE: {mae:.4f}, MSE: {mse:.4f}, RMSE: {rmse:.4f}")

Current MAE: 2.8190, MSE: 13.3720, RMSE: 3.6568
